<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Dailychallenge_2_W7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pinecone Serverless Reranking in Action

This notebook demonstrates how to use Pinecone's serverless reranking capabilities, including loading documents, executing reranking models, setting up a serverless index for medical notes, and comparing initial search results with reranked results.

## Part 1: Load Documents & Execute Reranking Model

### 1. Install Pinecone libraries

Run this command to install the necessary Pinecone client and notebook helper packages.

In [2]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 8.1 MB/s eta 0:00:00


### 2. Authenticate with Pinecone

This code block will prompt you for your Pinecone API key if it's not already set as an environment variable. Ensure you have your API key from your Pinecone dashboard.

In [39]:
import os
import re
from google.colab import userdata
import getpass
from pinecone import Pinecone, ServerlessSpec

# --- CONFIGURATION ---
# Si votre secret Colab a un nom différent, modifiez-le ici
NOM_DU_SECRET = 'PINECONE_API_KEY'

def clean_api_key(key):
    if not key: return None
    # Supprime les espaces, sauts de ligne et TOUS les caractères non-alphanumériques (sauf les tirets)
    key = key.strip()
    key = re.sub(r'[^\w\-]', '', key)
    return key

# 1. Récupération et nettoyage ultra-strict
try:
    raw_key = userdata.get(NOM_DU_SECRET)
    api_key = clean_api_key(raw_key)
    print(f"✅ Tentative avec le secret Colab : '{NOM_DU_SECRET}'")
except Exception:
    print(f"🔑 Secret '{NOM_DU_SECRET}' non trouvé.")
    api_key_input = getpass.getpass("Veuillez coller votre clé API Pinecone ici : ")
    api_key = clean_api_key(api_key_input)

# 2. Initialisation et validation immédiate
if api_key:
    os.environ["PINECONE_API_KEY"] = api_key
    pc = Pinecone(api_key=api_key)

    index_name = 'medical-notes-index'
    spec = ServerlessSpec(cloud='aws', region='us-east-1')

    try:
        # Test réel de la clé
        pc.list_indexes()
        print(f"🚀 Authentification réussie ! Client prêt.")
    except Exception as e:
        print(f"❌ La clé semble invalide (Erreur 401). Détails : {e}")
else:
    print("❌ Erreur : Aucune clé n'a pu être configurée.")

🔑 Secret 'PINECONE_API_KEY' non trouvé.
Veuillez coller votre clé API Pinecone ici : ··········
🚀 Authentification réussie ! Client prêt.


### 3. Instantiate the Pinecone client

The `pc` object will be your entry point for all Pinecone operations.

In [35]:
from pinecone import Pinecone
import os

# On récupère la clé nettoyée depuis l'environnement
api_key = os.environ.get("PINECONE_API_KEY")

if not api_key:
    print("❌ Erreur : PINECONE_API_KEY non trouvée.")
    pc = None
else:
    # Initialisation avec la clé propre
    pc = Pinecone(api_key=api_key)
    print("✅ Client Pinecone ré-initialisé avec succès.")

✅ Client Pinecone ré-initialisé avec succès.


### 4. Define your query & documents

We'll use a mix of documents related to the 'apple' fruit and 'Apple' company to test the reranker's contextual understanding.

In [36]:
query = "Tell me about Apple's products"
documents = [
    "Apples are a type of fruit that grow on trees and are often red or green.",
    "Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide.",
    "An apple a day keeps the doctor away is a common saying about the health benefits of eating apples.",
    "The iPhone, iPad, and MacBook are some of Apple's most popular products.",
    "Granny Smith apples are known for their tart flavor and crisp texture."
]

### 5. Call the reranker

We will use the `bge-reranker-v2-m3` model and request the top 3 results.

In [56]:
from pinecone import Pinecone
import os

# Exécution du Reranking (Partie 1)
try:
    # Transformation en liste de dictionnaires avec champ 'text'
    docs_to_rerank = [{"text": doc} for doc in documents]

    # Appel API
    response = pc.inference.rerank(
        model="bge-reranker-v2-m3",
        query=query,
        documents=docs_to_rerank,
        top_n=3
    )

    # Stockage pour la cellule d'affichage
    reranked = response

    # Vérification flexible du résultat
    results = getattr(response, 'data', getattr(response, 'results', None))

    if results is not None:
        print(f"✅ Reranking réussi. {len(results)} résultats trouvés.")
    else:
        print("⚠️ L'API a répondu mais le format est inattendu.")
        print(f"DEBUG: Type de réponse : {type(response)}")
        print(f"DEBUG: Contenu : {response}")
except Exception as e:
    print(f"❌ Erreur lors du reranking : {e}")

✅ Reranking réussi. 3 résultats trouvés.


### 6. Inspect reranked results

This function will display the reranked documents along with their scores, showing how the reranker reordered the results based on relevance to the query.

In [58]:
def show_reranked_results(query, response):
    print(f"Query: {query}")

    # Extraction des résultats peu importe le nom du champ (data ou results)
    matches = getattr(response, 'data', getattr(response, 'results', None))

    if not matches:
        print("⚠️ Aucun résultat trouvé dans la réponse du Reranker.")
        return

    for i, m in enumerate(matches):
        # Accès au texte du document
        text = "N/A"
        if hasattr(m, 'document'):
            doc = m.document
            text = doc.get('text', 'N/A') if isinstance(doc, dict) else getattr(doc, 'text', 'N/A')

        print(f"{str(i+1).rjust(4)}. Score: {m.score:.4f}, Text: {text}")

# Appel avec l'objet de réponse complet
if 'reranked' in locals() and reranked:
    show_reranked_results(query, reranked)
else:
    print("❌ La variable 'reranked' n'est pas définie.")

Query: Tell me about Apple's products
   1. Score: 0.9404, Text: The iPhone, iPad, and MacBook are some of Apple's most popular products.
   2. Score: 0.8213, Text: Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide.
   3. Score: 0.0162, Text: Apples are a type of fruit that grow on trees and are often red or green.


## Part 2: Setup a Serverless Index for Medical Notes

### 1. Install data & model libraries

These libraries are needed for loading, embedding, and manipulating medical note data.

In [7]:
!pip install pandas torch transformers

### 2. Import modules & define environment settings

We'll define the cloud, region, and index name for our serverless Pinecone index.

In [22]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Configuration de l'environnement
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')
spec = ServerlessSpec(cloud=cloud, region=region)
index_name = 'medical-notes-index'

# Initialisation du client si nécessaire (sécurité)
api_key = os.environ.get('PINECONE_API_KEY', '').strip()
if api_key:
    pc = Pinecone(api_key=api_key)
    print("✅ Client Pinecone et variables de configuration prêts.")

✅ Client Pinecone et variables de configuration prêts.


### 3. Create or recreate the index

We will create a new Pinecone index with the specified dimension and distance metric. The dimension of 384 matches the embedding model we will use.

In [43]:
# S'assurer que les variables globales sont présentes
index_name = 'medical-notes-index'

# Suppression si existant
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Création de l'index
pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec
)

# Attente de disponibilité
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

print(f"✅ Index '{index_name}' créé et prêt.")

✅ Index 'medical-notes-index' créé et prêt.


## Part 3: Load the Sample Data

### 1. Download & read JSONL

We will download sample medical notes data from a GitHub raw URL.

In [14]:
import requests
import tempfile
import os
import pandas as pd
import numpy as np

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "medical_notes.jsonl")
    url = "https://raw.githubusercontent.com/pinecone-io/examples/master/learn/search/reranking/medical_notes.jsonl"

    try:
        print(f"Téléchargement depuis : {url}")
        response = requests.get(url)
        response.raise_for_status()
        with open(file_path, "wb") as f:
            f.write(response.content)
        df = pd.read_json(file_path, orient='records', lines=True)
        print(f"✅ {len(df)} notes chargées.")
    except Exception as e:
        print(f"⚠️ Utilisation de données simulées avec vecteurs: {e}")
        # Création de données factices avec la dimension attendue (384)
        df = pd.DataFrame([
            {"id": "m1", "values": np.random.uniform(-1, 1, 384).tolist(), "metadata": {"text": "Patient has mild cough", "dept": "ER"}},
            {"id": "m2", "values": np.random.uniform(-1, 1, 384).tolist(), "metadata": {"text": "High blood pressure noted", "dept": "Cardiology"}},
            {"id": "m3", "values": np.random.uniform(-1, 1, 384).tolist(), "metadata": {"text": "Knee pain after fall", "dept": "Orthopedics"}}
        ])

Téléchargement depuis : https://raw.githubusercontent.com/pinecone-io/examples/master/learn/search/reranking/medical_notes.jsonl
⚠️ Utilisation de données simulées avec vecteurs: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/pinecone-io/examples/master/learn/search/reranking/medical_notes.jsonl


### 2. Preview the DataFrame

Display the head and shape of the DataFrame to ensure the data is loaded correctly.

In [44]:
# Preview sécurisé du DataFrame
if 'df' in locals():
    print("Data shape:", df.shape)
    display(df.head())
else:
    print("❌ Le DataFrame 'df' n'a pas été généré. Vérifiez la cellule de téléchargement.")

Data shape: (3, 3)


,id,values,metadata
0,m1,"[-0.17967426877696324, -0.5577912056330252, -0...","{'text': 'Patient has mild cough', 'dept': 'ER'}"
1,m2,"[0.8948124632770016, 0.6064556125684095, -0.16...","{'text': 'High blood pressure noted', 'dept': ..."
2,m3,"[-0.3371796859025664, 0.1683808382929355, 0.29...","{'text': 'Knee pain after fall', 'dept': 'Orth..."


## Part 4: Upsert Data into the Index

### 1. Instantiate index client & upsert

We will now push all medical note embeddings and their metadata into the Pinecone index.

In [45]:
# Initialisation explicite de l'index
index_name = 'medical-notes-index'
index = pc.Index(name=index_name)

# Upsert avec vérification du DataFrame
if 'df' in locals():
    index.upsert_from_dataframe(df)
    print("✅ Données insérées avec succès.")
else:
    print("❌ Erreur : DataFrame non trouvé.")

sending upsert requests:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Données insérées avec succès.


### 2. Wait for availability

This loop ensures that the upserted vectors are fully indexed and available for querying before proceeding.

In [46]:
# On s'assure que l'objet index est bien accessible
index = pc.Index(name=index_name)

def is_fresh(idx_obj):
    stats = idx_obj.describe_index_stats()
    return stats.total_vector_count > 0

print("⏳ Attente de l'indexation des vecteurs...")
while not is_fresh(index):
    time.sleep(2)

print("✅ Index prêt !")
display(index.describe_index_stats())

⏳ Attente de l'indexation des vecteurs...
✅ Index prêt !


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 3}},
 'total_vector_count': 3,
 'vector_type': 'dense'}

## Part 5: Query & Embedding Function

### 1. Define your embedding function

This function converts an input question into an embedding vector using the `sentence-transformers/all-MiniLM-L6-v2` model, allowing it to be compared with the indexed medical notes.

In [23]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    # Chargement local pour éviter les délais réseau répétés
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
        # Mean Pooling
        embedding = model_output.last_hidden_state[0].mean(dim=0)
    return embedding

print("✅ Fonction get_embedding définie.")

✅ Fonction get_embedding définie.


### 2. Run a semantic search query

We will perform a semantic search against our Pinecone index using a medical question and retrieve the top results.

In [47]:
# Recherche sémantique
question = "patient has severe chest pain and difficulty breathing"

# Appel de la fonction définie précédemment
query_vector = get_embedding(question).tolist()

# Recherche dans l'index
results = index.query(vector=query_vector, top_k=3, include_metadata=True)
sorted_matches = results['matches']
print("✅ Recherche sémantique effectuée.")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Recherche sémantique effectuée.


## Part 6: Display & Rerank Clinical Notes

### 1. Display initial search results

This function will display the initial search results, showing the ID, score, and metadata for each match.

In [48]:
def show_results(q, matches):
    print(f"Question: '{q}'")
    for i, match in enumerate(matches):
        text = match['metadata'].get('text', 'N/A')
        print(f"{str(i+1).rjust(4)}. ID: {match['id']} | Score: {match['score']:.4f} | Text: {text}")

show_results(question, sorted_matches)

Question: 'patient has severe chest pain and difficulty breathing'
   1. ID: m2 | Score: 0.0102 | Text: High blood pressure noted
   2. ID: m1 | Score: 0.0032 | Text: Patient has mild cough
   3. ID: m3 | Score: -0.0896 | Text: Knee pain after fall


### 2. Prepare documents for reranking

We transform the search results into a format suitable for the reranker, concatenating metadata into a `reranking_field`.

In [49]:
# Préparation pour le Reranking
transformed_documents = [
    {
        'id': match['id'],
        'text': match['metadata'].get('text', 'N/A')
    }
    for match in sorted_matches
]
print(f"✅ {len(transformed_documents)} documents préparés pour le reranking.")

✅ 3 documents préparés pour le reranking.


### 3. Execute serverless reranking

Now, we'll apply the reranking model with a more specific query to reorder the documents based on their content and metadata.

In [57]:
# Reranking final (Partie 6)
refined_query = "patient diagnosed with acute bronchitis"

try:
    # Reranking sur les notes médicales
    response = pc.inference.rerank(
        model="bge-reranker-v2-m3",
        query=refined_query,
        documents=transformed_documents,
        top_n=3
    )

    reranked_results = response

    # Extraction sécurisée
    results = getattr(response, 'data', getattr(response, 'results', None))

    if results is not None:
        print(f"✅ Reranking Pinecone Serverless terminé ({len(results)} docs).")
    else:
        print("⚠️ Reranking terminé mais aucun résultat n'a été extrait.")
        print(f"DEBUG: Réponse brute : {response}")
except Exception as e:
    print(f"❌ Erreur lors du reranking final : {e}")

✅ Reranking Pinecone Serverless terminé (3 docs).


### 4. Show reranked results

Display the reranked results to observe how the relevance ordering has changed after applying the reranker.

In [59]:
# Affichage final mis à jour
def show_final_reranked(q, response):
    print(f"Refined Query: '{q}'")

    matches = getattr(response, 'data', getattr(response, 'results', None))

    if not matches:
        print("⚠️ Aucun résultat renvoyé par le Reranker.")
        return

    for i, m in enumerate(matches):
        text = "N/A"
        if hasattr(m, 'document'):
            doc = m.document
            text = doc.get('text', 'N/A') if isinstance(doc, dict) else getattr(doc, 'text', 'N/A')

        print(f"{str(i+1).rjust(4)}. Score: {m.score:.4f} | Text: {text}")

if 'reranked_results' in locals() and reranked_results:
    show_final_reranked(refined_query, reranked_results)
else:
    print("❌ L'objet 'reranked_results' est manquant.")

Refined Query: 'patient diagnosed with acute bronchitis'
   1. Score: 0.0433 | Text: Patient has mild cough
   2. Score: 0.0036 | Text: High blood pressure noted
   3. Score: 0.0000 | Text: Knee pain after fall


### 5. Clean up (optional)

It's good practice to delete your index when you're done to avoid unnecessary charges.

In [52]:
# Nettoyage final sécurisé
if 'pc' in locals() and 'index_name' in locals():
    try:
        pc.delete_index(name=index_name)
        print(f"🗑️ Index '{index_name}' supprimé.")
    except Exception as e:
        print(f"ℹ️ L'index était peut-être déjà supprimé : {e}")

🗑️ Index 'medical-notes-index' supprimé.
